In [1]:
import pandas as pd
import geopandas as gpd
import pyogrio
import time

def extract_in_chunks():
    print("🌲 산불 발생 좌표 기준 임상도 추출 (무한루프 버그 수정 버전) 🌲")
    
    # 1. 파일 경로 설정 (아까 만든 공간DB를 입력으로 사용)
    gdb_path = r'd:\farm-system-public-02\jsw\data\임상도\임상도(15000).gdb'
    fire_csv_path = r'd:\farm-system-public-02\jsw\data\산불발생위치도_전국\산불_공간DB_위경도.csv'
    output_csv = r'd:\farm-system-public-02\jsw\data\임상도\임상도(15000).gdb\산불_공간DB_임상도2023_전국.csv'

    print(f"\n1. 산불 공간DB 로딩 중...")
    fire_df = pd.read_csv(fire_csv_path, encoding='utf-8-sig') 
    
    fire_gdf = gpd.GeoDataFrame(
        fire_df, 
        geometry=gpd.points_from_xy(fire_df['경도'], fire_df['위도']),
        crs="EPSG:4326"
    )
    
    layer_name = pyogrio.list_layers(gdb_path)[0][0]
    
    # ★ 핵심: 총 폴리곤 개수를 미리 파악하여 무한루프 차단
    total_features = pyogrio.read_info(gdb_path, layer=layer_name)['features']
    print(f"   -> 임상도 전체 폴리곤 개수: {total_features:,}개")
    
    # 좌표계 확인을 위해 임상도 딱 1줄만 먼저 읽음
    sample_gdf = gpd.read_file(gdb_path, layer=layer_name, engine='pyogrio', rows=1)
    target_crs = sample_gdf.crs
    
    if fire_gdf.crs != target_crs:
        fire_gdf = fire_gdf.to_crs(target_crs)

    columns_to_read = [
        'FRTP_CD', 'FRTP_NM', 'KOFTR_GROU', 'KOFTR_NM', 
        'DMCLS_CD', 'DMCLS_NM', 'AGCLS_CD', 'AGCLS_NM', 'DNST_CD', 'DNST_NM'
    ]

    chunk_size = 100000 
    skip = 0
    matched_results = []
    
    print("\n2. 임상도 청크(Chunk) 스캔 시작...")
    start_time = time.time()
    
    # ★ while True 대신 명확하게 한계치(total_features)까지만 돌도록 수정
    while skip < total_features: 
        chunk_gdf = gpd.read_file(
            gdb_path, 
            layer=layer_name, 
            engine='pyogrio', 
            columns=columns_to_read,
            skip_features=skip,
            max_features=chunk_size
        )
        
        joined = gpd.sjoin(fire_gdf, chunk_gdf, how='inner', predicate='within')
        if not joined.empty:
            matched_results.append(joined)
            
        skip += chunk_size
        if skip > total_features:
            skip = total_features
            
        print(f"   -> 누적 {skip:,} / {total_features:,} 개 스캔 완료... (현재 매칭: {len(joined)}건)")

    print(f"\n3. 스캔 완료! (총 소요 시간: {time.time() - start_time:.1f}초)")
    
    # 4. 결과 병합
    if matched_results:
        all_matched = pd.concat(matched_results, ignore_index=True)
        all_matched = all_matched.drop_duplicates(subset=['fire_id'])
        
        # 원래의 화재 데이터에 Left Join (산림 밖에서 난 불도 보존)
        final_df = pd.merge(fire_df, all_matched[columns_to_read + ['fire_id']], on='fire_id', how='left')
    else:
        print("   -> 매칭된 데이터가 없습니다.")
        final_df = fire_df

    # 5. 영어 컬럼명을 직관적인 한국어로 변경
    rename_dict = {
        'FRTP_CD': '임상구분코드',
        'FRTP_NM': '임상구분',
        'KOFTR_GROU': '수종코드',
        'KOFTR_NM': '수종',
        'DMCLS_CD': '경급코드',
        'DMCLS_NM': '경급',
        'AGCLS_CD': '영급코드',
        'AGCLS_NM': '영급',
        'DNST_CD': '소밀도코드',
        'DNST_NM': '소밀도'
    }
    final_df = final_df.rename(columns=rename_dict)
    

    # 결과 저장
    final_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\n4. 저장 완료: {output_csv}")

extract_in_chunks()


🌲 산불 발생 좌표 기준 임상도 추출 (무한루프 버그 수정 버전) 🌲

1. 산불 공간DB 로딩 중...
   -> 임상도 전체 폴리곤 개수: 3,410,925개

2. 임상도 청크(Chunk) 스캔 시작...


C:\Users\swoo6\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


   -> 누적 100,000 / 3,410,925 개 스캔 완료... (현재 매칭: 142건)
   -> 누적 200,000 / 3,410,925 개 스캔 완료... (현재 매칭: 153건)
   -> 누적 300,000 / 3,410,925 개 스캔 완료... (현재 매칭: 298건)
   -> 누적 400,000 / 3,410,925 개 스캔 완료... (현재 매칭: 309건)
   -> 누적 500,000 / 3,410,925 개 스캔 완료... (현재 매칭: 242건)
   -> 누적 600,000 / 3,410,925 개 스캔 완료... (현재 매칭: 308건)
   -> 누적 700,000 / 3,410,925 개 스캔 완료... (현재 매칭: 214건)
   -> 누적 800,000 / 3,410,925 개 스캔 완료... (현재 매칭: 277건)
   -> 누적 900,000 / 3,410,925 개 스캔 완료... (현재 매칭: 142건)
   -> 누적 1,000,000 / 3,410,925 개 스캔 완료... (현재 매칭: 284건)
   -> 누적 1,100,000 / 3,410,925 개 스캔 완료... (현재 매칭: 175건)
   -> 누적 1,200,000 / 3,410,925 개 스캔 완료... (현재 매칭: 215건)
   -> 누적 1,300,000 / 3,410,925 개 스캔 완료... (현재 매칭: 747건)
   -> 누적 1,400,000 / 3,410,925 개 스캔 완료... (현재 매칭: 365건)
   -> 누적 1,500,000 / 3,410,925 개 스캔 완료... (현재 매칭: 364건)
   -> 누적 1,600,000 / 3,410,925 개 스캔 완료... (현재 매칭: 206건)
   -> 누적 1,700,000 / 3,410,925 개 스캔 완료... (현재 매칭: 232건)
   -> 누적 1,800,000 / 3,410,925 개 스캔 완료... (현재 매칭: 195건)
   -> 누적 1